In [1]:
import pandas as pd
from torch.utils.data import DataLoader, Dataset


In [2]:
column_headers = ["date","timestamp","open","high","low","close","volume"]
filename = "/mnt/c/Shaukat/code_repo/HighFrequencyTradingCoding/data/DAT_MT_AUDUSD_M1_202411.csv"
df = pd.read_csv(filename, names=column_headers)

# combine dates and timestamps into single datetime column
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['timestamp'], format='%Y.%m.%d %H:%M')

# drop date, timestamp and volume
df.drop(columns=["date", "timestamp", "volume"], inplace=True)

# Setting datetime as index
df.set_index("datetime", inplace=True)

# Sort dataframe by datetime
df.sort_values(by="datetime", inplace=True)

# Add price label
df['label'] = df.apply(lambda row: 1 if row['close'] > row['open'] else 0, axis=1)

# Remove Nans
print(f"len_df: {len(df)} before removing NA")
df.dropna(inplace=True)
print(f"len_df: {len(df)} after removing NA")


len_df: 29658 before removing NA
len_df: 29658 after removing NA


In [4]:
df.head(5)

,open,high,low,close,label
datetime,,,,,
2024-11-01 00:00:00,0.65759,0.65767,0.65757,0.65758,0
2024-11-01 00:01:00,0.65759,0.65759,0.65750,0.65754,0
2024-11-01 00:02:00,0.65754,0.65764,0.65754,0.65761,1
2024-11-01 00:03:00,0.65760,0.65760,0.65751,0.65751,0
2024-11-01 00:04:00,0.65753,0.65758,0.65751,0.65755,1


# DataLoader

Prepare DATASET and DATALOADER

https://pytorch.org/tutorials/beginner/basics/data_tutorial.html

A custom Dataset class must implement three functions: __init__, __len__, and __getitem__.
        

In [12]:
index = '2024-11-01 00:03:00'
df.loc[index]

open     0.65760
high     0.65760
low      0.65751
close    0.65751
label    0.00000
Name: 2024-11-01 00:03:00, dtype: float64

In [17]:
window_len = 5
print(df.index.get_loc(index))
df.iloc[df.index.get_loc(index): df.index.get_loc(index) + window_len]

3


,open,high,low,close,label
datetime,,,,,
2024-11-01 00:03:00,0.65760,0.65760,0.65751,0.65751,0
2024-11-01 00:04:00,0.65753,0.65758,0.65751,0.65755,1
2024-11-01 00:05:00,0.65755,0.65761,0.65752,0.65754,0
2024-11-01 00:06:00,0.65753,0.65760,0.65753,0.65755,1
2024-11-01 00:07:00,0.65754,0.65757,0.65753,0.65756,1


In [79]:
# Sequence to Sequence Dataset and Data Loader
# Read last five minutes and get the labels of the next five minutes
# x = [x1,x2,x3,x4,x5] where x1 in R4
# y = [y6,y7,y8,y9,y10]

class ForexSeq2Seq(Dataset):
    def __init__(self, csv_files, transform = None, input_window=2, output_window=5):
        # init class parameters
        self.input_window = input_window
        self.output_window = output_window
        self.transform = transform

        dataframes = []
        column_headers = ["date","timestamp","open","high","low","close","volume"]
        for filename in csv_files:
            print(f"reading {filename}")
            df = pd.read_csv(filename, names=column_headers)

            # combine dates and timestamps into single datetime column
            df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['timestamp'], format='%Y.%m.%d %H:%M')

            # drop date, timestamp and volume
            df.drop(columns=["date", "timestamp", "volume"], inplace=True)

            # Append dataframes
            dataframes.append(df)
        
        # get data
        self.data = pd.concat(dataframes)

        # Remove Nans
        print(f"len_df: {len(self.data)} before removing NA")
        self.data.dropna(inplace=True)
        print(f"len_df: {len(self.data)} after removing NA")

        # Sort dataframe by datetime
        self.data.sort_values(by="datetime", inplace=True)
        self.data.reset_index(drop=True, inplace=True)

        # Add price label
        self.data['label'] = self.data.apply(lambda row: 1 if row['close'] > row['open'] else 0, axis=1)

    def __len__(self):
        '''
        Make sure that we do not access data that do not contain complete input and output
        Given a total dataset of N rows: 
            The first sample starts at index 0 and includes rows up to input_window + output_window - 1.
            The last sample starts at index N - input_window - output_window.
            Thus, the total number of valid samples is: len(self.data) - self.input_window - self.output_window + 1
            Assume:
            N = 10 rows.
            input_window = 3.
            output_window = 2.
            Valid Samples:
            Index	Input Window Rows	Output Window Rows
            0           Rows [0, 1, 2]      Rows [3, 4]
            1	        Rows [1, 2, 3]	    Rows [4, 5]
            2	        Rows [2, 3, 4]	    Rows [5, 6]
            3	        Rows [3, 4, 5]	    Rows [6, 7]
            4	        Rows [4, 5, 6]	    Rows [7, 8]
            5	        Rows [5, 6, 7]	    Rows [8, 9]
        len = N - input_window - output_window + 1 = 10 - 3 - 2 + 1 = 6 i.e. Dataloader will not go beyond 6
        '''
        return len(self.data) - self.input_window - self.output_window + 1
    
    def __getitem__(self, idx):
        '''
        It will be passed integer index internally whose range is determined by __len__
        '''
        input_start = idx
        input_end = idx + self.input_window
        output_start = input_end
        output_end = output_start + self.output_window

        # Get the input sequence 
        input_seq = self.data.iloc[input_start:input_end][["open","high","low","close"]]
        output_seq = self.data.iloc[output_start:output_end][["label"]]


        # return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)
        # it seems like they are automatically converted to tensors
        return input_seq.values.tolist(), output_seq.values.tolist()

        



In [80]:
# Init dataset
list_of_files = ['/mnt/c/Shaukat/code_repo/HighFrequencyTradingCoding/data/DAT_MT_AUDUSD_M1_202411.csv']
input_window = 2
output_window=5

# Read the dataset
dataset = ForexSeq2Seq(csv_files=list_of_files, input_window=input_window, output_window=output_window)

reading /mnt/c/Shaukat/code_repo/HighFrequencyTradingCoding/data/DAT_MT_AUDUSD_M1_202411.csv
len_df: 29658 before removing NA
len_df: 29658 after removing NA


In [81]:
print(type(dataset))
print("-----")
print(dataset.__dict__.keys())
print("-----")
print(dataset.data.head(5))
print("-----")
print(type(dataset.data))

<class '__main__.ForexSeq2Seq'>
-----
dict_keys(['input_window', 'output_window', 'transform', 'data'])
-----
      open     high      low    close            datetime  label
0  0.65759  0.65767  0.65757  0.65758 2024-11-01 00:00:00      0
1  0.65759  0.65759  0.65750  0.65754 2024-11-01 00:01:00      0
2  0.65754  0.65764  0.65754  0.65761 2024-11-01 00:02:00      1
3  0.65760  0.65760  0.65751  0.65751 2024-11-01 00:03:00      0
4  0.65753  0.65758  0.65751  0.65755 2024-11-01 00:04:00      1
-----
<class 'pandas.core.frame.DataFrame'>


In [82]:
test_input = dataset.data.iloc[0:3][["open","high","low","close"]]
test_output = dataset.data.iloc[3:8][["datetime", "label"]]

In [83]:
test_input

,open,high,low,close
0,0.65759,0.65767,0.65757,0.65758
1,0.65759,0.65759,0.65750,0.65754
2,0.65754,0.65764,0.65754,0.65761


In [84]:
test_input.values.tolist()

[[0.65759, 0.65767, 0.65757, 0.65758],
 [0.65759, 0.65759, 0.6575, 0.65754],
 [0.65754, 0.65764, 0.65754, 0.65761]]

In [85]:
test_output

,datetime,label
3,2024-11-01 00:03:00,0
4,2024-11-01 00:04:00,1
5,2024-11-01 00:05:00,0
6,2024-11-01 00:06:00,1
7,2024-11-01 00:07:00,1


In [110]:
# Lets check the dataloader
dataloader = DataLoader(dataset, batch_size=5)

# iterate through dataloader
# for features, labels in dataloader:
#     break

In [111]:
# iterate through dataloader
for features, labels in dataloader:
    break

In [112]:
features

[[tensor([0.6576, 0.6576, 0.6575, 0.6576, 0.6575], dtype=torch.float64),
  tensor([0.6577, 0.6576, 0.6576, 0.6576, 0.6576], dtype=torch.float64),
  tensor([0.6576, 0.6575, 0.6575, 0.6575, 0.6575], dtype=torch.float64),
  tensor([0.6576, 0.6575, 0.6576, 0.6575, 0.6575], dtype=torch.float64)],
 [tensor([0.6576, 0.6575, 0.6576, 0.6575, 0.6575], dtype=torch.float64),
  tensor([0.6576, 0.6576, 0.6576, 0.6576, 0.6576], dtype=torch.float64),
  tensor([0.6575, 0.6575, 0.6575, 0.6575, 0.6575], dtype=torch.float64),
  tensor([0.6575, 0.6576, 0.6575, 0.6575, 0.6575], dtype=torch.float64)]]

In [113]:
labels

[[tensor([1, 0, 1, 0, 1])],
 [tensor([0, 1, 0, 1, 1])],
 [tensor([1, 0, 1, 1, 0])],
 [tensor([0, 1, 1, 0, 0])],
 [tensor([1, 1, 0, 0, 0])]]

In [114]:
len(features)

2

In [115]:
len(labels)

5

In [117]:
features

[tensor([0.6576, 0.6575, 0.6576, 0.6575, 0.6575], dtype=torch.float64),
 tensor([0.6576, 0.6576, 0.6576, 0.6576, 0.6576], dtype=torch.float64),
 tensor([0.6575, 0.6575, 0.6575, 0.6575, 0.6575], dtype=torch.float64),
 tensor([0.6575, 0.6576, 0.6575, 0.6575, 0.6575], dtype=torch.float64)]